In [9]:
import pandas as pd
from typing import List
from chronos import Chronos2Pipeline
import numpy as np
DATA_DIR = "../../seeds/data/raw/aws_clean_baseline.parquet"
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2")
df = pd.read_parquet(DATA_DIR)
df.tail(20)

Loading weights: 100%|██████████| 170/170 [00:00<00:00, 5388.12it/s]


,timestamp,station_id,name,lat,lon,elevation_m,temp_c,humidity_pct,pressure_hpa
263140,2024-12-31 04:00:00,GAU001,Guwahati (Borjhar),26.106,91.585,54.0,14.1,93,1010.2
263141,2024-12-31 05:00:00,GAU001,Guwahati (Borjhar),26.106,91.585,54.0,14.1,92,1010.9
263142,2024-12-31 06:00:00,GAU001,Guwahati (Borjhar),26.106,91.585,54.0,13.7,98,1012.1
263143,2024-12-31 07:00:00,GAU001,Guwahati (Borjhar),26.106,91.585,54.0,15.6,94,1013.0
263144,2024-12-31 08:00:00,GAU001,Guwahati (Borjhar),26.106,91.585,54.0,17.0,89,1013.8
263145,2024-12-31 09:00:00,GAU001,Guwahati (Borjhar),26.106,91.585,54.0,18.3,86,1014.2
263146,2024-12-31 10:00:00,GAU001,Guwahati (Borjhar),26.106,91.585,54.0,20.2,79,1013.0
263147,2024-12-31 11:00:00,GAU001,Guwahati (Borjhar),26.106,91.585,54.0,22.1,69,1011.3
263148,2024-12-31 12:00:00,GAU001,Guwahati (Borjhar),26.106,91.585,54.0,23.2,62,1009.9
263149,2024-12-31 13:00:00,GAU001,Guwahati (Borjhar),26.106,91.585,54.0,23.7,59,1008.6


In [14]:
df.temp_c.values[-20:len(df.temp_c.values)-1]

array([14.1, 14.1, 13.7, 15.6, 17. , 18.3, 20.2, 22.1, 23.2, 23.7, 23.7,
       23.1, 20.8, 19.8, 19.2, 18.9, 17.4, 16.3, 15.6])

In [ ]:
df.temp_c.values[-1]

np.float64(15.4)

In [17]:
N=20
inputs = np.array(
        [
            [
                df.temp_c.values[-N:len(df.temp_c.values)-1],
                df.humidity_pct.values[-N:len(df.humidity_pct)-1],
                df.pressure_hpa.values[-N:len(df.pressure_hpa)-1],
            ]
        ]
    )
inputs

array([[[  14.1,   14.1,   13.7,   15.6,   17. ,   18.3,   20.2,   22.1,
           23.2,   23.7,   23.7,   23.1,   20.8,   19.8,   19.2,   18.9,
           17.4,   16.3,   15.6],
        [  93. ,   92. ,   98. ,   94. ,   89. ,   86. ,   79. ,   69. ,
           62. ,   59. ,   57. ,   59. ,   69. ,   72. ,   78. ,   79. ,
           86. ,   91. ,   92. ],
        [1010.2, 1010.9, 1012.1, 1013. , 1013.8, 1014.2, 1013. , 1011.3,
         1009.9, 1008.6, 1007.8, 1007.6, 1007.6, 1008.3, 1008.9, 1009.7,
         1009.9, 1010. , 1010. ]]])

In [18]:
quantiles, mean = pipeline.predict_quantiles(
        inputs, prediction_length=1, quantile_levels=[0.05, 0.5, 0.95]
    )

In [21]:
forecast = quantiles[0].tolist()

In [62]:
actual = [12,df.humidity_pct.values[-1],1000]
actual

[12, np.int64(92), 1000]

In [63]:
for i,reading in enumerate(forecast):
    p5,p50,p95 = reading[0]
    corridor_width = p95 - p5
    upper_breach = np.maximum(0.0, actual[i] - p95)
    lower_breach = np.maximum(0.0, p5 - actual[i])
    total_breach = upper_breach + lower_breach
    severity = total_breach / corridor_width
    severity = np.round(severity, 4)
    print(severity)
    predicted_anomly = severity > 0.8
    print(predicted_anomly)
    

0.9639
True
0.0
False
7.3163
True
